# 02 — Data Cleaning dan DuckDB SUMMARIZE

**Milestone 2 | Analisis Big Data**

Tujuan: memeriksa missing values, tanggal gagal parse, duplikasi judul di sumber & tanggal sama, serta outlier panjang teks. **Tidak mengubah dataset asli.** Data hasil olahan berupa **tabel analisis terpilih**, bukan salinan seluruh artikel atau pengganti arsip sumber.

Proses ini menggunakan Polars Lazy API dan `sink_parquet`. DuckDB digunakan untuk SQL `SUMMARIZE` setelah data hasil olahan tersimpan.

## 1. Library dan dataset mentah

In [1]:
from pathlib import Path
import sys
import polars as pl

# Isi manual hanya jika lokasi data berbeda dari struktur repository.
# Contoh Windows: r"C:\Users\NamaUser\bigdata_teman\data\raw\indonesian-political-news-clean"
# Contoh Colab setelah mount Drive: "/content/drive/MyDrive/bigdata_teman/data/raw/indonesian-political-news-clean"
DATASET_DIR_OVERRIDE = None

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

possible_dirs = [
    ROOT / "data" / "raw" / "indonesian-political-news-clean",
    Path.home() / "bigdata_teman" / "data" / "raw" / "indonesian-political-news-clean",
    Path("/content/drive/MyDrive/bigdata_teman/data/raw/indonesian-political-news-clean"),
]

if DATASET_DIR_OVERRIDE:
    DATA_DIR = Path(DATASET_DIR_OVERRIDE)
else:
    DATA_DIR = next((p for p in possible_dirs if p.exists()), possible_dirs[0])

PARQUET_FILES = sorted(DATA_DIR.rglob("*.parquet")) if DATA_DIR.exists() else []
if not PARQUET_FILES:
    raise FileNotFoundError(
        f"File Parquet tidak ditemukan di {DATA_DIR}. "
        "Sesuaikan DATASET_DIR_OVERRIDE pada cell ini atau ikuti data/README.md."
    )
PARQUET_PATHS = [str(p) for p in PARQUET_FILES]
print("Root repo:", ROOT)
print("Folder dataset:", DATA_DIR)
print("Jumlah file Parquet:", len(PARQUET_FILES))
print("Ukuran:", round(sum(f.stat().st_size for f in PARQUET_FILES)/1e9,3), "GB")
print("Versi Polars:", pl.__version__)

Root repo: C:\Users\raihan bhakti\bigdata_teman
Folder dataset: C:\Users\raihan bhakti\bigdata_teman\data\raw\indonesian-political-news-clean
Jumlah file Parquet: 22
Ukuran: 5.446 GB
Versi Polars: 1.44.2


In [2]:
import duckdb
import json
from datetime import datetime, timezone

# Bila data mentah dari Google Drive (Colab), letakkan hasil cleaning
# di folder proyek pada Drive agar notebook 03 dapat membacanya di sesi baru.
OUTPUT_ROOT_OVERRIDE = None
if OUTPUT_ROOT_OVERRIDE:
    OUTPUT_ROOT = Path(OUTPUT_ROOT_OVERRIDE)
elif str(DATA_DIR).startswith("/content/drive/"):
    OUTPUT_ROOT = DATA_DIR.parents[2]  # .../bigdata_teman/data/raw/nama_dataset
else:
    OUTPUT_ROOT = ROOT

PROCESSED_DIR = OUTPUT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_FILE = PROCESSED_DIR / "political_news_analysis.parquet"
OUTPUT_TABLES = OUTPUT_ROOT / "output" / "tables"
OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)

required = ["id", "news_title", "news_text", "news_hostname", "news_date", "news_kabkot", "news_author"]
raw_all = pl.scan_parquet(PARQUET_PATHS)
found = raw_all.collect_schema().names()
assert all(x in found for x in required), f"Kolom yang hilang: {sorted(set(required)-set(found))}"
raw = raw_all.select(required)
print("Kolom yang digunakan:", required)

Kolom yang digunakan: ['id', 'news_title', 'news_text', 'news_hostname', 'news_date', 'news_kabkot', 'news_author']


## 2. Normalisasi awal dan quality flags

Judul dibersihkan dari spasi berlebih untuk kunci duplikasi. Tanggal diubah menjadi `Date` dengan sepuluh karakter awal. Kolom `news_kabkot` merupakan label geografis yang ada di dataset, **bukan bukti lokasi media atau pusat kejadian yang terverifikasi**.

In [3]:
prepared = (
    raw.with_columns([
        pl.col("id").cast(pl.String),
        pl.col("news_title").cast(pl.String).fill_null("").str.strip_chars().alias("news_title"),
        pl.col("news_text").cast(pl.String).fill_null("").alias("news_text"),
        pl.col("news_hostname").cast(pl.String).fill_null("tidak diketahui")
          .str.strip_chars().str.to_lowercase().alias("news_hostname"),
        pl.col("news_kabkot").cast(pl.String).fill_null("").str.strip_chars().alias("news_kabkot"),
        pl.col("news_author").cast(pl.String).fill_null("").str.strip_chars().alias("news_author"),
        pl.col("news_date").cast(pl.String).fill_null("").alias("news_date"),
    ])
    .with_columns([
        pl.col("news_date").str.slice(0, 10)
          .str.strptime(pl.Date, "%Y-%m-%d", strict=False).alias("publication_date"),
        pl.col("news_title").str.len_chars().alias("title_chars"),
        pl.col("news_text").str.len_chars().alias("body_chars"),
        pl.col("news_title").str.to_lowercase()
          .str.replace_all(r"\s+", " ").str.strip_chars().alias("title_key"),
    ])
)

## 3. Audit kualitas sebelum cleaning

In [4]:
audit = prepared.select([
    pl.len().alias("baris_mentah"),
    (pl.col("publication_date").is_null()).sum().alias("tanggal_invalid"),
    (pl.col("title_chars") < 5).sum().alias("judul_terlalu_pendek"),
    (pl.col("body_chars") < 50).sum().alias("isi_terlalu_pendek"),
    (pl.col("news_kabkot") == "").sum().alias("wilayah_kosong"),
    (pl.col("news_author") == "").sum().alias("penulis_kosong"),
]).collect()
print(audit)

shape: (1, 6)
┌──────────────┬─────────────────┬──────────────────────┬────────────────────┬────────────────┬────────────────┐
│ baris_mentah ┆ tanggal_invalid ┆ judul_terlalu_pendek ┆ isi_terlalu_pendek ┆ wilayah_kosong ┆ penulis_kosong │
│ ---          ┆ ---             ┆ ---                  ┆ ---                ┆ ---            ┆ ---            │
│ u32          ┆ u32             ┆ u32                  ┆ u32                ┆ u32            ┆ u32            │
╞══════════════╪═════════════════╪══════════════════════╪════════════════════╪════════════════╪════════════════╡
│ 1628288      ┆ 84332           ┆ 20                   ┆ 0                  ┆ 0              ┆ 149514         │
└──────────────┴─────────────────┴──────────────────────┴────────────────────┴────────────────┴────────────────┘


## 4. Aturan cleaning

Aturan yang dipilih: wajib tanggal dapat diparse, panjang judul minimal 5 karakter, dan panjang isi minimal 50 karakter. Angka ini merupakan aturan operasional analisis yang bisa direvisi berdasarkan distribusi aktual, bukan standar nasional. **Nilai wilayah/penulis kosong tidak menyebabkan baris dihapus**, melainkan diberi kategori `Tidak tersedia`. Duplikasi ditentukan dari host + judul normal + hari terbit. Risiko artikel berbeda dengan kunci sama wajib dicatat.

In [5]:
from datetime import date

# Batas tahun untuk analisis temporal
DATE_MIN = date(2000, 1, 1)
DATE_MAX = date(2026, 12, 31)

# Filter kualitas wajib
valid = (
    prepared.filter(
        pl.col("publication_date").is_not_null()
        & (pl.col("title_chars") >= 5)
        & (pl.col("body_chars") >= 50)
    )
    .with_columns([
        pl.when(pl.col("news_kabkot") == "")
          .then(pl.lit("Tidak tersedia"))
          .otherwise(pl.col("news_kabkot"))
          .alias("news_kabkot"),

        pl.when(pl.col("news_author") == "")
          .then(pl.lit("Tidak tersedia"))
          .otherwise(pl.col("news_author"))
          .alias("news_author"),
    ])
)

# Menghitung data valid
valid_count = valid.select(pl.len()).collect().item()

# Menghapus duplikasi berdasarkan kunci
dedup = valid.unique(
    subset=["news_hostname", "title_key", "publication_date"],
    keep="first",
    maintain_order=False
)

# Menandai tanggal di luar batas pemeriksaan
dedup = dedup.with_columns([
    pl.col("publication_date")
      .alias("publication_date_original"),

    (~pl.col("publication_date").is_between(
        DATE_MIN, DATE_MAX
    )).alias("date_out_of_range"),
])

# Tanggal anomali menjadi null untuk analisis temporal
# Artikel tetap dipertahankan
dedup = dedup.with_columns(
    pl.when(pl.col("date_out_of_range"))
      .then(pl.lit(None, dtype=pl.Date))
      .otherwise(pl.col("publication_date"))
      .alias("publication_date")
)

# Menghitung hasil akhir
clean_count = dedup.select(pl.len()).collect().item()

raw_count = int(audit["baris_mentah"][0])

print("Baris mentah:", raw_count)
print("Baris valid sebelum dedup:", valid_count)
print("Dihapus aturan wajib:", raw_count - valid_count)
print("Dihapus karena duplikasi:", valid_count - clean_count)
print("Baris akhir:", clean_count)

print("Tanggal anomali yang ditandai:")
print(
    dedup.select(
        pl.col("date_out_of_range").sum()
    ).collect()
)

assert 0 < clean_count <= valid_count <= raw_count

Baris mentah: 1628288
Baris valid sebelum dedup: 1543943
Dihapus aturan wajib: 84345
Dihapus karena duplikasi: 286
Baris akhir: 1543657
Tanggal anomali yang ditandai:
shape: (1, 1)
┌───────────────────┐
│ date_out_of_range │
│ ---               │
│ u32               │
╞═══════════════════╡
│ 403               │
└───────────────────┘


## 5. Outlier panjang artikel

Gunakan ambang **Q3 + 3×IQR** untuk panjang isi. Outlier ditandai dengan boolean, tidak otomatis dibuang karena artikel panjang bisa saja sah. Data mentah teks penuh tidak disalin ke output agar Parquet analisis lebih ringan.

In [6]:
quantiles = dedup.select([
    pl.col("body_chars").quantile(0.25).alias("q1"),
    pl.col("body_chars").quantile(0.75).alias("q3"),
]).collect()

q1 = float(quantiles["q1"][0])
q3 = float(quantiles["q3"][0])

outlier_limit = q3 + 3 * (q3 - q1)

print("Q1:", q1)
print("Q3:", q3)
print("Batas outlier:", outlier_limit)

analysis_lf = (
    dedup
    .with_columns(
        (pl.col("body_chars") > outlier_limit)
        .alias("body_length_outlier")
    )
    .select([
        "id",
        "news_title",
        "news_hostname",
        "news_author",
        "news_kabkot",
        "publication_date",
        "publication_date_original",
        "date_out_of_range",
        "title_chars",
        "body_chars",
        "body_length_outlier"
    ])
)

print("Jumlah outlier panjang teks:")
print(
    analysis_lf
    .select(pl.col("body_length_outlier").sum())
    .collect()
)

Q1: 1007.0
Q3: 2482.0
Batas outlier: 6907.0
Jumlah outlier panjang teks:
shape: (1, 1)
┌─────────────────────┐
│ body_length_outlier │
│ ---                 │
│ u32                 │
╞═════════════════════╡
│ 6953                │
└─────────────────────┘


## 6. Simpan Parquet terolah secara streaming

In [7]:
if "con" in globals():
    con.close()

In [8]:
analysis_lf.sink_parquet(
    str(PROCESSED_FILE),
    compression="zstd"
)

assert PROCESSED_FILE.exists(), "Output Parquet tidak terbentuk."

print("Tersimpan di:", PROCESSED_FILE)
print("Ukuran output:", round(PROCESSED_FILE.stat().st_size / 1e6, 2), "MB")

print(
    "Baris hasil:",
    pl.scan_parquet(str(PROCESSED_FILE))
    .select(pl.len())
    .collect()
    .item()
)

Tersimpan di: C:\Users\raihan bhakti\bigdata_teman\data\processed\political_news_analysis.parquet
Ukuran output: 76.59 MB
Baris hasil: 1543657


In [9]:
import polars as pl

hasil = pl.scan_parquet(str(PROCESSED_FILE))

print("Kolom hasil cleaning:")
print(hasil.collect_schema().names())

print("\nValidasi tanggal dan outlier:")
print(
    hasil.select([
        pl.len().alias("total_baris"),
        pl.col("publication_date").null_count().alias("tanggal_null"),
        pl.col("date_out_of_range").sum().alias("tanggal_anomali"),
        pl.col("body_length_outlier").sum().alias("outlier_teks"),
        pl.col("publication_date").min().alias("tanggal_min"),
        pl.col("publication_date").max().alias("tanggal_max")
    ]).collect()
)

Kolom hasil cleaning:
['id', 'news_title', 'news_hostname', 'news_author', 'news_kabkot', 'publication_date', 'publication_date_original', 'date_out_of_range', 'title_chars', 'body_chars', 'body_length_outlier']

Validasi tanggal dan outlier:
shape: (1, 6)
┌─────────────┬──────────────┬─────────────────┬──────────────┬─────────────┬─────────────┐
│ total_baris ┆ tanggal_null ┆ tanggal_anomali ┆ outlier_teks ┆ tanggal_min ┆ tanggal_max │
│ ---         ┆ ---          ┆ ---             ┆ ---          ┆ ---         ┆ ---         │
│ u32         ┆ u32          ┆ u32             ┆ u32          ┆ date        ┆ date        │
╞═════════════╪══════════════╪═════════════════╪══════════════╪═════════════╪═════════════╡
│ 1543657     ┆ 403          ┆ 403             ┆ 6953         ┆ 2002-04-03  ┆ 2026-12-31  │
└─────────────┴──────────────┴─────────────────┴──────────────┴─────────────┴─────────────┘


## 7. DuckDB SQL — `SUMMARIZE`

DuckDB membaca langsung hasil Parquet, lalu menyusun ringkasan statistik setiap kolom. Ini berbeda dari profiling Polars, sehingga kedua library digunakan sesuai perannya.

In [10]:
import duckdb

# Membuat koneksi DuckDB
con = duckdb.connect()

# Membaca Parquet hasil cleaning
con.read_parquet(
    str(PROCESSED_FILE)
).create_view("news_clean", replace=True)

# Menjalankan profiling DuckDB
summary = con.sql("""
    SUMMARIZE news_clean
""").df()

display(summary)

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,id,VARCHAR,1,999999,1762198,None,None,None,None,None,1543657,0.00
1,news_title,VARCHAR,! Maaf Data Tidak Ditemukan - Jambi Prima - In...,🪨 Produksi Batu Bara 2025 dan 2026 Diproyeksi ...,1537550,None,None,None,None,None,1543657,0.00
2,news_hostname,VARCHAR,1kabar.com,zuper.fajar.co.id,5091,None,None,None,None,None,1543657,0.00
3,news_author,VARCHAR,$YM,𝗗𝟯𝗹 𝟬𝟮,46377,None,None,None,None,None,1543657,0.00
4,news_kabkot,VARCHAR,Aceh Barat,Yalimo,1044,None,None,None,None,None,1543657,0.00
5,publication_date,DATE,2002-04-03,2026-12-31,3127,2026-01-26 10:53:55.547745,None,2025-12-31,2026-02-10,2026-03-13,1543657,0.03
6,publication_date_original,DATE,0001-11-29,2569-03-20,3262,2026-01-15 11:42:02.194762,None,2025-12-31,2026-02-11,2026-03-13,1543657,0.00
7,date_out_of_range,BOOLEAN,false,true,2,None,None,None,None,None,1543657,0.00
8,title_chars,UINTEGER,5,4310,563,87.02095413683222,29.083461726756934,70,86,102,1543657,0.00
9,body_chars,UINTEGER,160,457008,7743,1881.2927651673915,1356.0614431786294,1008,1802,2482,1543657,0.00


In [11]:
hasil_validasi = con.sql("""
    SELECT
        COUNT(*) AS total_baris,

        COUNT(*) FILTER (
            WHERE publication_date < DATE '2000-01-01'
               OR publication_date >= DATE '2027-01-01'
        ) AS tanggal_anomali,

        COUNT(DISTINCT id) AS id_unik_aktual

    FROM news_clean
""").df()

display(hasil_validasi)

,total_baris,tanggal_anomali,id_unik_aktual
0,1543657,0,1543657


## 8. Query analitik DuckDB dan laporan kualitas

In [12]:
daily_sql = con.sql("""
SELECT DATE_TRUNC('month', publication_date)::DATE AS bulan,
       COUNT(*) AS jumlah_artikel
FROM news_clean
GROUP BY 1
ORDER BY 1
""").df()
display(daily_sql.head(12))

sources_sql = con.sql("""
SELECT news_hostname, COUNT(*) AS jumlah_artikel
FROM news_clean
GROUP BY 1
ORDER BY 2 DESC
LIMIT 10
""").df()
display(sources_sql)

,bulan,jumlah_artikel
0,2002-04-01,19
1,2003-04-01,14
2,2003-12-01,1
3,2004-08-01,1
4,2004-12-01,1
5,2005-01-01,2
6,2005-05-01,1
7,2005-08-01,1
8,2005-12-01,1
9,2006-04-01,1


,news_hostname,jumlah_artikel
0,tribunnews.com,124572
1,rri.co.id,87789
2,pikiran-rakyat.com,37583
3,antaranews.com,36607
4,jawapos.com,36399
5,disway.id,30890
6,detik.com,30049
7,kompas.com,18510
8,merdeka.com,13119
9,kompasiana.com,9066


In [13]:
outlier_count = (
    pl.scan_parquet(str(PROCESSED_FILE))
    .select(
        pl.col("body_length_outlier").sum()
    )
    .collect()
    .item()
)
report = {
    "dataset": "ardimardiana/indonesian-political-news-clean",
    "baris_awal": raw_count,
    "baris_tidak_memenuhi_syarat_wajib": raw_count - valid_count,
    "baris_duplikasi_kunci": valid_count - clean_count,
    "baris_akhir": clean_count,
    "wilayah_kosong_sebelum_cleaning": int(audit["wilayah_kosong"][0]),
    "penulis_kosong_sebelum_cleaning": int(audit["penulis_kosong"][0]),
    "tanggal_invalid": int(audit["tanggal_invalid"][0]),
    "batas_outlier_panjang_teks": round(outlier_limit,2),
    "outlier_panjang_teks_ditandai": int(outlier_count),
    "kebijakan_outlier": "ditandai, tidak dibuang",
    "tanggal_eksekusi_utc": datetime.now(timezone.utc).isoformat(),
}
report_path = OUTPUT_TABLES / "data_quality_report.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(report, ensure_ascii=False, indent=2))
print("Laporan tersimpan:", report_path)
con.close()

{
  "dataset": "ardimardiana/indonesian-political-news-clean",
  "baris_awal": 1628288,
  "baris_tidak_memenuhi_syarat_wajib": 84345,
  "baris_duplikasi_kunci": 286,
  "baris_akhir": 1543657,
  "wilayah_kosong_sebelum_cleaning": 0,
  "penulis_kosong_sebelum_cleaning": 149514,
  "tanggal_invalid": 84332,
  "batas_outlier_panjang_teks": 6907.0,
  "outlier_panjang_teks_ditandai": 6953,
  "kebijakan_outlier": "ditandai, tidak dibuang",
  "tanggal_eksekusi_utc": "2026-10-15T08:48:37.557488+00:00"
}
Laporan tersimpan: C:\Users\raihan bhakti\bigdata_teman\output\tables\data_quality_report.json


## Ringkasan Milestone 2

Hasil cleaning tersimpan di `data/processed/political_news_analysis.parquet` dan menjadi input `03_eda_and_insights.ipynb`. Perubahan kualitas data harus dilaporkan dari keluaran notebook, tidak menyalin angka contoh yang belum diverifikasi. Karena sumber sudah menyebut dirinya *cleaned*, tahap ini dipahami sebagai **audit serta standardisasi ulang untuk keperluan analisis**, bukan klaim bahwa data asal sepenuhnya kotor.